# 추가 실험 1 — 245문항 1회 실행 (추론 끔 · T = 0)

목적: 생성 코드의 **정답률**과 **실행 오류율**만 본다. 반복·조건 비교 없음.

| 항목 | 값 |
|---|---|
| 문항 | `questions_245.csv` 245건 (기준 코드가 gold와 전건 일치) |
| 반복 | **1회** (문항당 1콜, 총 245콜) |
| 모델 | `gpt-6-luna` (OpenAI Chat Completions) |
| 추론 | **끔** (`reasoning_effort="none"`) |
| 온도 | **0.0** |
| max_completion_tokens | 300 |
| 파이프라인 | 프롬프트 → 첫 `return`까지 절단 → PP 1회 → 실행 (SamEval과 동일) |

`lab/` 폴더에서 이 노트북을 열고 위에서부터 실행하세요. 호출 실패·중단 시 같은 셀을 다시 실행하면 이어서 돕니다(재개 키 = 조건·문항·회차).

In [1]:
import pathlib, sys, inspect
import numpy as np, pandas as pd

# 이 노트북은 lab/ 폴더 안에서 연다고 가정한다 (exp1_lib.py 가 있는 폴더).
LAB = pathlib.Path.cwd()
if not (LAB / "exp1_lib.py").exists():
    raise SystemExit(f"lab/ 에서 여세요. 현재 폴더에 exp1_lib.py 없음: {LAB}")

sys.path.insert(0, str(LAB))
import exp1_lib as E

# 테이블 parquet 위치. exp1_lib 기본 경로가 없으면 repo/data/tables 로 대체.
TABLE_DIR = E.DATA
if not TABLE_DIR.exists():
    TABLE_DIR = E.REPO / "data" / "tables"
E.DATA = TABLE_DIR

print("LAB       :", LAB)
print("TABLE_DIR :", TABLE_DIR, "| 존재:", TABLE_DIR.exists())
print("HARNESS   :", E.HARNESS_VERSION)
print("run_pilot :", str(inspect.signature(E.run_pilot))[:110])

LAB       : c:\Users\seude\Desktop\dp4agent\lab
TABLE_DIR : C:\Users\seude\Desktop\dp4agent\data\tables | 존재: True
HARNESS   : prefill_emul_v2
run_pilot : (client, questions: 'pd.DataFrame', ref: 'pd.DataFrame', out_csv: 'pathlib.Path', condition: 'str', n_rep: 'in


In [2]:
qs  = pd.read_csv(LAB / "questions_245.csv", encoding="utf-8-sig")
ref = pd.read_csv(LAB / "answers_ref_245.csv", encoding="utf-8-sig")

# 질문 파일에는 n 열이 없다 → 파일 순서 = n (answers_ref_245 의 n 과 1:1)
qs["n"] = np.arange(1, len(qs) + 1)

assert len(qs) == 245, f"문항 수 {len(qs)}"
assert len(ref) == 245, f"기준표 {len(ref)}"
assert (qs["n"].to_numpy() == ref["n"].to_numpy()).all(), "n 정렬 불일치"
assert (ref["key"].str.rsplit("|", n=2).str[0].to_numpy() == qs["dataset"].to_numpy()).all(), "dataset 순서 불일치"
assert ref["answer_ref"].notna().all(), "기준 정답 결측"

qs = qs.merge(ref[["n", "answer_ref"]], on="n", how="left", validate="one_to_one")
print("문항:", qs.shape, "| 기준표 gold 일치:", int((ref["match"] == "O").sum()), "/", len(ref))
qs[["n", "dataset", "question", "answer_ref"]].head(3)

문항: (245, 10) | 기준표 gold 일치: 245 / 245


,n,dataset,question,answer_ref
0,1,001_Forbes,What is the age of the youngest billionaire?,19.0
1,2,001_Forbes,How many billionaires are there from the 'Tech...,343.0
2,3,001_Forbes,What's the total worth of billionaires in the ...,583600.0


In [3]:
client = E.make_client()

MODEL  = E.MODEL_DEFAULT          # gpt-6-luna
COND   = "R245_eNone_T0"          # 조건 라벨 (CSV 식별자)
TEMP   = 0.0
MAXTOK = 300
EFFORT = "none"

# 이 조합(추론 끔 + T=0)이 실제로 열려 있는지 1콜로 확인
try:
    t = E.call_llm(client, "say ok", MODEL, TEMP, 16, EFFORT)
    print("설정 확인 OK |", t["finish_reason"], "|", repr(t["text"][:40]))
except Exception as e:
    raise SystemExit(f"이 조합이 거부됨 → {type(e).__name__}: {e}")

설정 확인 OK | stop | 'ok'


In [4]:
# 스모크 테스트: 3문항만 먼저 (별도 CSV, 본런과 충돌 없음)
smoke = E.run_pilot(client, qs.head(3), ref, "run245_T0_smoke.csv", condition=COND,
                    n_rep=1, model=MODEL, temperature=TEMP, max_tokens=MAXTOK,
                    reasoning_effort=EFFORT, verbose=True)
print()
print(smoke[["n", "correct", "exec_ok", "exec_err", "finish_reason", "body_len"]].to_string(index=False))

  R245_eNone_T0 n=  1 rep=1 O exec=ok fr=stop     out=54 rea=0
  R245_eNone_T0 n=  2 rep=1 O exec=ok fr=stop     out=47 rea=0
  R245_eNone_T0 n=  3 rep=1 O exec=ok fr=stop     out=80 rea=0

 n  correct  exec_ok  exec_err finish_reason  body_len
 1     True     True       NaN          stop       199
 2     True     True       NaN          stop       182
 3     True     True       NaN          stop       302


In [5]:
OUT = "run245_T0.csv"   # lab/result/ 아래에 저장

runs = E.run_pilot(client, qs, ref, OUT, condition=COND, n_rep=1,
                   model=MODEL, temperature=TEMP, max_tokens=MAXTOK,
                   reasoning_effort=EFFORT, verbose=True)

print()
print("저장:", E._out(OUT))
print("행 수:", len(runs), "| 문항 수:", runs["n"].nunique())
runs = runs[runs["condition"] == COND].copy()

  R245_eNone_T0 n=  1 rep=1 O exec=ok fr=stop     out=53 rea=0
  R245_eNone_T0 n=  2 rep=1 O exec=ok fr=stop     out=55 rea=0
  R245_eNone_T0 n=  3 rep=1 O exec=ok fr=stop     out=80 rea=0
  R245_eNone_T0 n=  4 rep=1 O exec=ok fr=stop     out=66 rea=0
  R245_eNone_T0 n=  5 rep=1 O exec=ok fr=stop     out=126 rea=0
  R245_eNone_T0 n=  6 rep=1 O exec=ok fr=stop     out=57 rea=0
  R245_eNone_T0 n=  7 rep=1 O exec=ok fr=stop     out=34 rea=0
  R245_eNone_T0 n=  8 rep=1 O exec=ok fr=stop     out=51 rea=0
  R245_eNone_T0 n=  9 rep=1 O exec=ok fr=stop     out=45 rea=0
  R245_eNone_T0 n= 10 rep=1 O exec=ok fr=stop     out=57 rea=0
  R245_eNone_T0 n= 11 rep=1 O exec=ok fr=stop     out=60 rea=0
  R245_eNone_T0 n= 12 rep=1 O exec=ok fr=stop     out=89 rea=0
  R245_eNone_T0 n= 13 rep=1 O exec=ok fr=stop     out=45 rea=0
  R245_eNone_T0 n= 14 rep=1 O exec=ok fr=stop     out=46 rea=0
  R245_eNone_T0 n= 15 rep=1 O exec=ok fr=stop     out=61 rea=0
  R245_eNone_T0 n= 16 rep=1 O exec=ok fr=stop     out=

In [6]:
def report(d: pd.DataFrame, label: str = "") -> pd.DataFrame:
    ce   = d["call_error"].astype("string").fillna("")      # 호출 자체 실패
    called = d[ce == ""]
    ok   = called["exec_ok"].astype(bool)
    rows = [
        ("호출 수",               len(d)),
        ("호출 실패",             int((ce != "").sum())),
        ("실행 오류 건수",        int((~ok).sum())),
        ("실행 성공률 %",         round(ok.mean() * 100, 2) if len(called) else None),
        ("정답 건수",             int(called["correct"].sum())),
        ("정답률 % (전체)",       round(called["correct"].mean() * 100, 2) if len(called) else None),
        ("정답률 % (실행 성공만)", round(called.loc[ok, "correct"].mean() * 100, 2) if ok.any() else None),
        ("값 None 건수",          int(called["null_answer"].sum())),
        ("finish=length 건수",    int((called["finish_reason"] == "length").sum())),
        ("응답시간 평균 s",       round(called["latency_s"].mean(), 2) if len(called) else None),
    ]
    return pd.DataFrame(rows, columns=["지표", label or "값"])

report(runs, COND)

,지표,R245_eNone_T0
0,호출 수,245.00
1,호출 실패,0.00
2,실행 오류 건수,2.00
3,실행 성공률 %,99.18
4,정답 건수,234.00
5,정답률 % (전체),95.51
6,정답률 % (실행 성공만),96.30
7,값 None 건수,2.00
8,finish=length 건수,0.00
9,응답시간 평균 s,1.59


In [7]:
# 참고: 실행 실패의 예외 유형 (지표 아님, 진단용)
err = runs[runs["exec_err"].astype("string").fillna("") != ""].copy()
if len(err):
    err["etype"] = err["exec_err"].str.split(":").str[0]
    print(err["etype"].value_counts().rename_axis("예외").to_frame("건수").to_string())
else:
    print("실행 실패 없음")

             건수
예외             
SyntaxError   2


In [8]:
# 오답·실패 문항 목록 (확인용)
bad = runs[(~runs["exec_ok"].astype(bool)) | (~runs["correct"].astype(bool))]
bad[["n", "dataset", "answer_ref", "answer_agent", "exec_err"]].assign(
    body=bad["body"].str.slice(0, 80)
).to_string(index=False)

"  n           dataset  answer_ref  answer_agent                                            exec_err                                                                               body\n 17          004_Taxi 99807.00000      0.000000                                                 NaN      # ['PULocationID']\\n    # The airport area corresponds to pickup location ID \n 21        006_London   563.00000     33.000000                                                 NaN      # The columns used to answer the question: ['neighbourhood_cleansed']\\n    # \n108       029_NYTimes   147.00000    145.000000                                                 NaN      # The columns used to answer the question: ['headline']\\n    # The type of th\n112 030_Professionals   450.00000           NaN SyntaxError: '(' was never closed (<agent>, line 7)       # The columns used to answer the question: ['In which country do you current\n166          044_IMDb    96.07927     94.709941                             

In [9]:
summ = report(runs, COND)
p = E.RESULT_DIR / "run245_T0_summary.csv"
summ.to_csv(p, index=False, encoding="utf-8-sig")
print(p)
summ

C:\Users\seude\Desktop\dp4agent\lab\result\run245_T0_summary.csv


,지표,R245_eNone_T0
0,호출 수,245.00
1,호출 실패,0.00
2,실행 오류 건수,2.00
3,실행 성공률 %,99.18
4,정답 건수,234.00
5,정답률 % (전체),95.51
6,정답률 % (실행 성공만),96.30
7,값 None 건수,2.00
8,finish=length 건수,0.00
9,응답시간 평균 s,1.59


## 해석 시 주의

- **T=0 + 추론 끔**이므로 반복 변동은 측정 대상이 아니다. 여기서 나오는 실행 오류는 샘플링이 아니라 프롬프트 조립·후처리·코드 생성의 구조적 실패다.
- 245문항은 기준 코드가 gold와 **전건 일치**하는 집합이다. 즉 오답률에 "기준 자체가 틀린 문항"이 섞이지 않는다.
- `finish=length`가 많으면 300토큰 상한이 병목이다. 이 경우에만 상한을 올려 재확인한다.
- 실행 오류의 유형 분류(컬럼/단위/필터/집계)는 이 실행 이후에 별도로 정한다. 이 노트북은 건수만 센다.

In [10]:
import pathlib, sys, re, textwrap
import numpy as np, pandas as pd

LAB = pathlib.Path.cwd()
if not (LAB / "exp1_lib.py").exists():
    raise SystemExit(f"lab/ 폴더에서 실행하세요. 현재: {LAB}")
sys.path.insert(0, str(LAB))
import exp1_lib as E

if not E.DATA.exists():
    E.DATA = E.REPO / "data" / "tables"

SRC = E._out("run245_T0.csv")
print("입력:", SRC, "| 존재:", SRC.exists())


입력: C:\Users\seude\Desktop\dp4agent\lab\result\run245_T0.csv | 존재: True


In [11]:
d = pd.read_csv(SRC, encoding="utf-8-sig")
d = d[d["call_error"].astype("string").fillna("") == ""].copy()
print(f"재채점 대상 {len(d)}행")


재채점 대상 245행


In [12]:
def _body(raw, truncate=True):
    out = []
    for l in str(raw).split("\n"):
        out.append(l)
        if truncate and re.match(r"^\s*return\b", l):
            break
    return "\n".join(out).strip("\n")

def _src_pp(body, cols):
    return (f"def answer(df: pd.DataFrame):\n"
            f"    df.columns = {list(cols)!r}\n"
            f"{E._pp()(body)}\n")

def _src_no_pp(body, cols):
    return (f"def answer(df: pd.DataFrame):\n"
            f"    df.columns = {list(cols)!r}\n"
            f"{textwrap.indent(textwrap.dedent(body), '    ')}\n")

def _exec(body, df, cols, fn):
    if not body.strip():
        return None, "EmptyBody"
    try:
        ns = {"pd": pd, "np": np, "re": re}
        exec(compile(fn(body, cols), "<rescore>", "exec"), ns)
        v = ns["answer"](df.copy())
        if isinstance(v, (pd.Series, np.ndarray, list, tuple)):
            v = v[0] if len(v) else None
        if isinstance(v, np.generic):
            v = v.item()
        return v, None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"[:160]

_tab, rows = {}, []
for _, r in d.iterrows():
    ds = r["dataset"]
    if ds not in _tab:
        _tab[ds] = E._load_table(ds)
    df, cols = _tab[ds], list(_tab[ds].columns)
    body = _body(r["raw"])
    for name, fn in (("현행 (PP 1회)", _src_pp), ("PP 없음", _src_no_pp)):
        v, err = _exec(body, df, cols, fn)
        rows.append({"variant": name, "n": int(r["n"]), "exec_ok": err is None,
                     "err": err or "", "answer": v, "answer_ref": r["answer_ref"],
                     "correct": bool(E.close(v, r["answer_ref"])), "null": v is None})
cmp = pd.DataFrame(rows)
print(len(cmp), "행 재채점 완료")


490 행 재채점 완료


In [13]:
tab = []
for name, g in cmp.groupby("variant", sort=False):
    ok = g["exec_ok"].astype(bool)
    tab.append({
        "변형": name,
        "호출 수": len(g),
        "실행 오류 건수": int((~ok).sum()),
        "실행 성공률 %": round(ok.mean() * 100, 2),
        "값 None 건수": int(g["null"].sum()),
        "값 None %": round(g["null"].mean() * 100, 2),
        "정답 건수": int(g["correct"].sum()),
        "정답률 %": round(g["correct"].mean() * 100, 2),
        "정답률 % (실행 성공만)": round(g.loc[ok, "correct"].mean() * 100, 2) if ok.any() else None,
    })
tab = pd.DataFrame(tab)

cmp.to_csv(E.RESULT_DIR / "run245_T0_pp_rows.csv", index=False, encoding="utf-8-sig")
tab.to_csv(E.RESULT_DIR / "run245_T0_pp_compare.csv", index=False, encoding="utf-8-sig")
print("저장:", E.RESULT_DIR / "run245_T0_pp_compare.csv")
tab


저장: C:\Users\seude\Desktop\dp4agent\lab\result\run245_T0_pp_compare.csv


,변형,호출 수,실행 오류 건수,실행 성공률 %,값 None 건수,값 None %,정답 건수,정답률 %,정답률 % (실행 성공만)
0,현행 (PP 1회),245,2,99.18,2,0.82,234,95.51,96.30
1,PP 없음,245,7,97.14,7,2.86,229,93.47,96.22
